### Gas Cylinder Evaluation

Datasets evaluated:
- `gas_bottles` — CylinDeRS, single class `gas_cylinder`
- `gas_cylinder` — Roboflow, classes `Cap_Close`, `Cap_open`
- `gas_hack` — Roboflow, single class `gas`

For each dataset:
- Load `best.pt` from the matching training run
- Run `model.val()` on validation and test once each
- Save metrics to CSV
- Save a handful of annotated test images
- Compare training-time and standalone-evaluation metrics

Outputs: per-dataset evaluation CSVs, combined summary, annotated examples.

In [1]:
from pathlib import Path
import pandas as pd
import torch
from ultralytics import YOLO
import yaml

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.14.1+cpu
CUDA available: False


In [2]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = ROOT / "data"
RUNS_DIR = ROOT / "results" / "gas_bottle_training"
OUTPUT_DIR = ROOT / "results" / "gas_bottle_evaluation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_CANDIDATES = [
    "gas_bottles",
    "gas_cylinder",
    "gas_hack",
]

def load_yaml(ds_dir):
    for name in ["data.yaml", "data_ultralytics.yaml"]:
        p = ds_dir / name
        if p.exists():
            return p, yaml.safe_load(p.read_text())
    return None, None

device = 0 if torch.cuda.is_available() else "cpu"

print(f"Data directory:   {DATA_DIR.relative_to(ROOT)}")
print(f"Runs directory:   {RUNS_DIR.relative_to(ROOT)}")
print(f"Output directory: {OUTPUT_DIR.relative_to(ROOT)}")
print(f"Device:           {device}")

Data directory:   data
Runs directory:   results\gas_bottle_training
Output directory: results\gas_bottle_evaluation
Device:           cpu


### Verify available runs

List every dataset that has both a `data.yaml` and a `best.pt`.

In [3]:
available = []

for name in DATASET_CANDIDATES:
    ds = DATA_DIR / name
    run_dir = RUNS_DIR / f"yolo11n_{name}"
    best_pt = run_dir / "weights" / "best.pt"
    yaml_path, cfg = load_yaml(ds) if ds.exists() else (None, None)

    ok = ds.exists() and yaml_path is not None and best_pt.exists()
    print(f"{name:15s} dataset={ds.exists()}  yaml={yaml_path.name if yaml_path else None}  best.pt={best_pt.exists()}  ->  {'OK' if ok else 'SKIP'}")

    if ok:
        available.append({
            "name": name,
            "dataset_dir": ds,
            "yaml_path": yaml_path,
            "cfg": cfg,
            "run_dir": run_dir,
            "best_pt": best_pt,
        })

print(f"\nDatasets ready for evaluation: {[a['name'] for a in available]}")

gas_bottles     dataset=True  yaml=data.yaml  best.pt=True  ->  OK
gas_cylinder    dataset=True  yaml=data.yaml  best.pt=True  ->  OK
gas_hack        dataset=True  yaml=data.yaml  best.pt=True  ->  OK

Datasets ready for evaluation: ['gas_bottles', 'gas_cylinder', 'gas_hack']


### Configuration

Same image size and batch size as the training runs, so metrics are comparable.

In [4]:
IMGSZ = 416
BATCH = 4
CONF = 0.25
MAX_ANNOTATED = 5

print(f"imgsz={IMGSZ}  batch={BATCH}  conf={CONF}  max_annotated={MAX_ANNOTATED}")

imgsz=416  batch=4  conf=0.25  max_annotated=5


### Run validation and test evaluation

For every available dataset, load its `best.pt`, run `model.val()` on the validation split, then on the test split. Save both metric rows.

In [5]:
eval_rows = []

for entry in available:
    name = entry["name"]
    yaml_path = entry["yaml_path"]
    best_pt = entry["best_pt"]
    run_dir = entry["run_dir"]

    print("\n" + "=" * 60)
    print(f"Evaluating {name}  ->  {best_pt.relative_to(ROOT)}")
    print("=" * 60)

    model = YOLO(str(best_pt))

    for split in ["val", "test"]:
        metrics = model.val(
            data=str(yaml_path),
            split=split,
            imgsz=IMGSZ,
            batch=BATCH,
            device=device,
            project=str(OUTPUT_DIR),
            name=f"{name}_{split}",
            exist_ok=True,
            plots=True,
            verbose=False,
        )

        row = {
            "dataset": name,
            "split": split,
            "precision": float(metrics.box.mp),
            "recall": float(metrics.box.mr),
            "mAP50": float(metrics.box.map50),
            "mAP50-95": float(metrics.box.map),
        }
        eval_rows.append(row)

        print(f"{name:15s} {split:5s}  P={row['precision']:.4f}  R={row['recall']:.4f}  mAP50={row['mAP50']:.4f}  mAP50-95={row['mAP50-95']:.4f}")

df_eval = pd.DataFrame(eval_rows)
EVAL_CSV = OUTPUT_DIR / "evaluation_metrics.csv"
df_eval.to_csv(EVAL_CSV, index=False)
print(f"\nSaved evaluation metrics to {EVAL_CSV.relative_to(ROOT)}")


Evaluating gas_bottles  ->  results\gas_bottle_training\yolo11n_gas_bottles\weights\best.pt
Ultralytics 8.4.174  Python-3.11.9 torch-2.14.1+cpu CPU (AMD Ryzen 7 6800HS with Radeon Graphics)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access  (ping: 0.30.2 ms, read: 5.57.7 MB/s, size: 3.4 KB)
val: Scanning D:\BCS Files\Y3S1\AI Applications MBI38j\Labs\primagaz-yolo\data\gas_bottles\valid\labels.cache... 4808 images, 0 backgrounds, 54 corrupt: 100% ━━━━━━━━━━━━ 4862/4862  0.0s
val: D:\BCS Files\Y3S1\AI Applications MBI38j\Labs\primagaz-yolo\data\gas_bottles\valid\images\1_JPG_jpg.rf.0081d2a73451cf687fedfb424962b0d0.jpg: ignoring corrupt image/label: image size (10, 4) <10 pixels
val: D:\BCS Files\Y3S1\AI Applications MBI38j\Labs\primagaz-yolo\data\gas_bottles\valid\images\1_JPG_jpg.rf.0c8ad52afaf9da159ce4f7f034767351.jpg: ignoring corrupt image/label: image size (38, 9) <10 pixels
val: D:\BCS Files\Y3S1\AI Applications MBI38j\Labs\p

### Compare training-time vs standalone evaluation

Reads each run's `training_summary.csv` and puts them next to the validation and test results.

In [6]:
train_rows = []

for entry in available:
    name = entry["name"]
    run_dir = entry["run_dir"]
    summary_path = run_dir / "training_summary.csv"

    if not summary_path.exists():
        print(f"[skip] {name}: no training_summary.csv")
        continue

    s = pd.read_csv(summary_path).iloc[0]
    train_rows.append({
        "dataset": name,
        "split": "train",
        "precision": s.get("precision"),
        "recall": s.get("recall"),
        "mAP50": s.get("mAP50"),
        "mAP50-95": s.get("mAP50-95"),
    })

df_train = pd.DataFrame(train_rows)
df_all = pd.concat([df_train, df_eval], ignore_index=True)

COMPARE_CSV = OUTPUT_DIR / "evaluation_comparison.csv"
df_all.to_csv(COMPARE_CSV, index=False)

for name in df_all["dataset"].unique():
    print("\n" + "-" * 60)
    print(name)
    print("-" * 60)
    sub = df_all[df_all["dataset"] == name][["split", "precision", "recall", "mAP50", "mAP50-95"]]
    print(sub.to_string(index=False))

print(f"\nSaved comparison to {COMPARE_CSV.relative_to(ROOT)}")


------------------------------------------------------------
gas_bottles
------------------------------------------------------------
split  precision   recall    mAP50  mAP50-95
train   0.981250 0.761730 0.872580  0.835590
  val   0.916958 0.686356 0.886616  0.842143
 test   0.932140 0.684468 0.895238  0.854545

------------------------------------------------------------
gas_cylinder
------------------------------------------------------------
split  precision   recall    mAP50  mAP50-95
train   0.789410 0.310810 0.345010  0.126240
  val   0.789558 0.310811 0.344935  0.126276
 test   0.834374 0.342105 0.377492  0.156696

------------------------------------------------------------
gas_hack
------------------------------------------------------------
split  precision  recall    mAP50  mAP50-95
train   0.003330     1.0 0.835800  0.590040
  val   0.003333     1.0 0.930073  0.708758
 test   0.003338     1.0 0.909286  0.651750

Saved comparison to results\gas_bottle_evaluation\evaluation

### Save annotated example predictions

For each dataset, run the model on a few test images and save the annotated versions.

In [7]:
import cv2

EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

for entry in available:
    name = entry["name"]
    ds = entry["dataset_dir"]
    best_pt = entry["best_pt"]

    test_images_dir = ds / "test" / "images"
    test_images = sorted(
        p for p in test_images_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in EXTS
    ) if test_images_dir.exists() else []

    if not test_images:
        print(f"[skip] {name}: no test images at {test_images_dir}")
        continue

    sample = test_images[:MAX_ANNOTATED]
    out_dir = OUTPUT_DIR / "annotated" / name
    out_dir.mkdir(parents=True, exist_ok=True)

    model = YOLO(str(best_pt))

    for img_path in sample:
        res = model.predict(
            source=str(img_path),
            conf=CONF,
            imgsz=IMGSZ,
            device=device,
            verbose=False,
        )[0]

        annotated = res.plot()
        out_path = out_dir / f"pred_{img_path.stem}.jpg"
        cv2.imwrite(str(out_path), annotated)

        n = 0 if res.boxes is None else len(res.boxes)
        print(f"{name:15s} {img_path.name:35s} detections={n}")

print(f"\nAnnotated images saved under {OUTPUT_DIR.relative_to(ROOT)}/annotated/")

gas_bottles     -7-_png_jpg.rf.fa231b88048ce280d54562b688630131.jpg detections=1
gas_bottles     -8-_png_jpg.rf.bbc5693caeb3b26e460574fd54071cdd.jpg detections=1
gas_bottles     101_jpg.rf.13308acea8d4e1d2f3594a50f500a907.jpg detections=1
gas_bottles     101_jpg.rf.2d5ebb0da241cecb05647a1ea731eff0.jpg detections=0
gas_bottles     105_jpg.rf.b6d4887972b91ec385e9de1d42ce3598.jpg detections=1
gas_cylinder    gas_with_cap-114-_jpg.rf.3ba11965f728e01e750b9ae03297edbc.jpg detections=1
gas_cylinder    gas_with_cap-119-_jpg.rf.51a192f973b6b5f6f8a3e63fe472cb59.jpg detections=1
gas_cylinder    gas_with_cap-128-_jpg.rf.c133f3cafdada4015dd4b4c68a09a717.jpg detections=1
gas_cylinder    gas_with_cap-129-_jpg.rf.4640542a05108bd0909448b3a6a20790.jpg detections=1
gas_cylinder    gas_with_cap-149-_jpg.rf.ff7b0aa5e90c96dea8f1b3fb51297e83.jpg detections=1
gas_hack        20220213_155521_011_jpg.rf.227c691d36192d951f0c9af881eae957.jpg detections=0
gas_hack        20220213_155521_023_jpg.rf.0cae58d3e36b92d1

### Overview

Single table with all metrics for every dataset and split.

In [8]:
pivot = df_all.pivot_table(
    index="dataset",
    columns="split",
    values=["precision", "recall", "mAP50", "mAP50-95"],
    aggfunc="first",
)

print("=" * 70)
print("EVALUATION OVERVIEW")
print("=" * 70)
print(pivot.to_string())
print("=" * 70)
print(f"\nFiles:")
print(f"  {EVAL_CSV.relative_to(ROOT)}")
print(f"  {COMPARE_CSV.relative_to(ROOT)}")
print(f"  {OUTPUT_DIR.relative_to(ROOT)}/annotated/")

EVALUATION OVERVIEW
                 mAP50                     mAP50-95                    precision                       recall                   
split             test    train       val      test    train       val      test    train       val      test    train       val
dataset                                                                                                                         
gas_bottles   0.895238  0.87258  0.886616  0.854545  0.83559  0.842143  0.932140  0.98125  0.916958  0.684468  0.76173  0.686356
gas_cylinder  0.377492  0.34501  0.344935  0.156696  0.12624  0.126276  0.834374  0.78941  0.789558  0.342105  0.31081  0.310811
gas_hack      0.909286  0.83580  0.930073  0.651750  0.59004  0.708758  0.003338  0.00333  0.003333  1.000000  1.00000  1.000000

Files:
  results\gas_bottle_evaluation\evaluation_metrics.csv
  results\gas_bottle_evaluation\evaluation_comparison.csv
  results\gas_bottle_evaluation/annotated/


In [ ]:
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
lbl_dir = ROOT / "data" / "gas_hack" / "train" / "labels"

files = sorted(lbl_dir.glob("*.txt"))
print(f"Label files: {len(files)}")
for f in files[:3]:
    print("=" * 40)
    print(f.name)
    print(f.read_text()[:300])

import yaml
y = ROOT / "data" / "gas_hack" / "data.yaml"
print("=" * 40)
print(y.read_text())

### CVAT Custom Dataset Evaluation

In [ ]:
cvat_run = ROOT / "results" / "cvat_batch_1" / "training" / "yolo11n_cvat_batch_1"
cvat_yaml = ROOT / "data" / "cvat_batch_1_split" / "data.yaml"
cvat_best = cvat_run / "weights" / "best.pt"

if cvat_best.exists() and cvat_yaml.exists():
    m = YOLO(str(cvat_best))
    r = m.val(data=str(cvat_yaml), split="val", imgsz=416, batch=4, device=device, verbose=False)
    print(f"cvat_batch_1 val: P={r.box.mp:.4f} R={r.box.mr:.4f} mAP50={r.box.map50:.4f} mAP50-95={r.box.map:.4f}")
else:
    print("CVAT run not found, skip.")

Ultralytics 8.4.174  Python-3.11.9 torch-2.14.1+cpu CPU (AMD Ryzen 7 6800HS with Radeon Graphics)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access  (ping: 0.20.1 ms, read: 782.5385.8 MB/s, size: 4869.8 KB)
val: Scanning D:\BCS Files\Y3S1\AI Applications MBI38j\Labs\primagaz-yolo\data\cvat_batch_1_split\valid\labels.cache... 25 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 25/25  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 2.3it/s 3.0s0.5s
                   all         25         36      0.995      0.972      0.975      0.877
Speed: 0.5ms preprocess, 27.7ms inference, 0.0ms loss, 2.1ms postprocess per image
Results saved to D:\BCS Files\Y3S1\AI Applications MBI38j\Labs\primagaz-yolo\notebooks\runs\detect\val-5
cvat_batch_1 val: P=0.9945 R=0.9722 mAP50=0.9750 mAP50-95=0.8772
